#02_silver_eventos.sql

Camada Silver - fact_eventos (assistencial)

Este notebook transforma eventos assistenciais (bronze.eventos_assistenciais) em uma fact table Silver tipada, validada e deduplicada para análises de utilização, custo e sinistralidade.

##Objetivos
- Tipa campos: data_evento (TIMESTAMP), valores (DECIMAL)
- Padroniza textos (tipo_evento, cid_grupo, rede_atendimento)
- Deduplica por evento_id (registro mais recente por ingestion_ts)
- Aplica regras de qualidade: evento_id, beneficiario_id obrigatórios, data_evento parseável, valor_cobrado e valor_pago não negativos (quando presentes)
- Trata inconsistências sem perder dado útil: se valor_pago > valor_cobrado, manter registro e marcar flag_valor_inconsistente = 1
- Persiste idempotente via MERGE (Delta) usando row_hash
- Roda incremental via checkpoint (silver_ops.pipeline_checkpoint)

##Estratégia:
- SQL-first com TEMP VIEWs
- TRY_TO_TIMESTAMP / TRY_CAST para conversões resilientes
- row_number() para dedupe determinístico
- rejects em tabela dedicada com reject_reason
- flags para inconsistências não-bloqueantes

##Chave natural

evento_id

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Pipeline checkpoint

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.pipeline_checkpoint (
  table_name STRING,
  last_ingestion_ts TIMESTAMP,
  last_run_ts TIMESTAMP,
  status STRING
)
USING DELTA;

##Tabela de destino (silver + rejects)

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_eventos (
  evento_id STRING,
  beneficiario_id STRING,
  data_evento TIMESTAMP,
  tipo_evento STRING,
  cid_grupo STRING,
  prestador_id STRING,
  rede_atendimento STRING,
  valor_cobrado DECIMAL(18,2),
  valor_pago DECIMAL(18,2),

  -- flags úteis
  flag_valor_inconsistente INT,
  flag_valor_pago_negativo INT,
  flag_valor_cobrado_negativo INT,

  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_eventos (
  evento_id STRING,
  beneficiario_id STRING,
  data_evento STRING,
  tipo_evento STRING,
  cid_grupo STRING,
  prestador_id STRING,
  rede_atendimento STRING,
  valor_cobrado STRING,
  valor_pago STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Leitura incremental do bronze (via checkpoint)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_eventos_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_eventos'
)
SELECT *
FROM healthcare_dev.bronze.eventos_assistenciais
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem e padronização

In [0]:
CREATE OR REPLACE TEMP VIEW stg_eventos_typed AS
SELECT
  cast(evento_id as string) AS evento_id,
  cast(beneficiario_id as string) AS beneficiario_id,

  try_to_timestamp(data_evento) AS data_evento,

  upper(trim(cast(tipo_evento as string))) AS tipo_evento,
  upper(trim(cast(cid_grupo as string))) AS cid_grupo,
  cast(prestador_id as string) AS prestador_id,
  upper(trim(cast(rede_atendimento as string))) AS rede_atendimento,

  try_cast(valor_cobrado as decimal(18,2)) AS valor_cobrado,
  try_cast(valor_pago as decimal(18,2)) AS valor_pago,

  ingestion_ts,
  load_id,
  source_file
FROM stg_eventos_raw;

##Deduplicação determinística (evento_id)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_eventos_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY evento_id
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_eventos_typed
) x
WHERE rn = 1;

##Regras de qualidade + Flag (classificação)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_eventos_classified AS
SELECT
  *,

  -- flags corp (não-bloqueantes)
  CASE WHEN valor_pago IS NOT NULL AND valor_pago < 0 THEN 1 ELSE 0 END AS flag_valor_pago_negativo,
  CASE WHEN valor_cobrado IS NOT NULL AND valor_cobrado < 0 THEN 1 ELSE 0 END AS flag_valor_cobrado_negativo,

  CASE
    WHEN valor_pago IS NOT NULL AND valor_cobrado IS NOT NULL AND valor_pago > valor_cobrado THEN 1
    ELSE 0
  END AS flag_valor_inconsistente,

  -- Reject para problemas estruturais
  CASE
    WHEN evento_id IS NULL OR evento_id = '' THEN 'MISSING_EVENTO_ID'
    WHEN beneficiario_id IS NULL OR beneficiario_id = '' THEN 'MISSING_BENEFICIARIO_ID'
    WHEN data_evento IS NULL THEN 'INVALID_DATA_EVENTO'
    ELSE NULL
  END AS reject_reason
FROM stg_eventos_dedup;

##Persiste rejects

In [0]:
INSERT INTO healthcare_dev.silver_rejects.fact_eventos
SELECT
  evento_id,
  beneficiario_id,
  cast(data_evento as string) AS data_evento,
  tipo_evento,
  cid_grupo,
  prestador_id,
  rede_atendimento,
  cast(valor_cobrado as string) AS valor_cobrado,
  cast(valor_pago as string) AS valor_pago,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_eventos_classified
WHERE reject_reason IS NOT NULL;


##Valid + row_hash (MERGE idempotente)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_eventos_valid AS
SELECT
  evento_id,
  beneficiario_id,
  data_evento,
  tipo_evento,
  cid_grupo,
  prestador_id,
  rede_atendimento,
  valor_cobrado,
  valor_pago,
  flag_valor_inconsistente,
  flag_valor_pago_negativo,
  flag_valor_cobrado_negativo,
  ingestion_ts,
  load_id,
  source_file,

  sha2(concat_ws('||',
    evento_id,
    coalesce(beneficiario_id,''),
    coalesce(cast(data_evento as string),''),
    coalesce(tipo_evento,''),
    coalesce(cid_grupo,''),
    coalesce(prestador_id,''),
    coalesce(rede_atendimento,''),
    coalesce(cast(valor_cobrado as string),''),
    coalesce(cast(valor_pago as string),''),
    coalesce(cast(flag_valor_inconsistente as string),''),
    coalesce(cast(flag_valor_pago_negativo as string),''),
    coalesce(cast(flag_valor_cobrado_negativo as string),'')
  ), 256) AS row_hash
FROM stg_eventos_classified
WHERE reject_reason IS NULL;

##MERGE na silver

In [0]:
MERGE INTO healthcare_dev.silver.fact_eventos AS tgt
USING stg_eventos_valid AS src
ON tgt.evento_id = src.evento_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualiza checkpoint

In [0]:
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_eventos' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status
FROM stg_eventos_valid;

##Checa sanidade

In [0]:
SELECT COUNT(*) AS n_silver_eventos
FROM healthcare_dev.silver.fact_eventos;

In [0]:
SELECT COUNT(*) AS n_rejects_eventos
FROM healthcare_dev.silver_rejects.fact_eventos;

In [0]:
SELECT reject_reason, COUNT(*) AS qtd
FROM healthcare_dev.silver_rejects.fact_eventos
GROUP BY reject_reason
ORDER BY qtd DESC;

In [0]:
SELECT
  ROUND(100.0 * SUM(CASE WHEN flag_valor_inconsistente = 1 THEN 1 ELSE 0 END) / COUNT(*), 4) AS pct_valor_inconsistente
FROM healthcare_dev.silver.fact_eventos;

In [0]:
SELECT
  COUNT(*) AS total,
  SUM(flag_valor_pago_negativo) AS n_valor_pago_negativo,
  SUM(flag_valor_cobrado_negativo) AS n_valor_cobrado_negativo,
  ROUND(100.0 * SUM(flag_valor_pago_negativo) / COUNT(*), 4) AS pct_valor_pago_negativo
FROM healthcare_dev.silver.fact_eventos;